In [1]:
import os
import json
import chromadb

from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown
from chromadb.config import Settings
from getpass import getpass


load_dotenv(override=True)



True

In [2]:
chroma_client = chromadb.PersistentClient(path="./")

collection = chroma_client.get_or_create_collection(name="restaurant_vector_db")

In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_KEY"),
)

# client = OpenAI(
#     base_url="https://ollama.com/v1",
#     api_key=os.getenv("ollama_key"),
# )

MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
# MODEL = "llama3-chatqa"
# EMBEDDING_MODEL = "nvidia/nemotron-3.5-embed-8b:free"

In [4]:
def ask(messages, temperature=0):
    try:
        r = client.chat.completions.create(model=MODEL, messages=messages, temperature=temperature)
        return (r.choices[0].message.content or "").strip()
    except Exception as e:                    # no key? rate limit (429)? wrong model name?
        return f"ERROR: {e}"

EMB_MODEL = "nvidia/nemotron-3-embed-1b:free"      # embedding model (free, Day 24)
# EMB_MODEL = "nomic-embed-text-v1_5"
# EMB_MODEL = "nomic-embed-text-v2-moe"
def embed(texts):
    r = client.embeddings.create(model=EMB_MODEL, input=texts)    # one request for the whole list
    vectors = []
    for item in r.data:
        vectors.append(item.embedding)                            # one vector per text
    return vectors   

## Load + chunk

The same code as Day 30. The documents are in `chai_wala_docs`.

In [5]:
import re
import json
from pathlib import Path

SECTION = re.compile(r"^\d+\.\s+\S")           # "1. Restaurant Overview"
BRANCH  = re.compile(r"^Branch \d+\s*---")      # "Branch 1 --- Gulshan-e-Iqbal (Flagship)"

def load_documents(path):
    p = Path(path)
    files = [p] if p.is_file() else sorted(p.glob("*.txt"))   # file ya folder, dono chalega
    return [{"source": f.name, "text": f.read_text(encoding="utf-8")} for f in files]

def chunk_by_section(text):
    chunks, parent, heading, buf = [], "", "", []

    def flush():
        body = "\n".join(buf).strip()
        buf.clear()
        if len(body) <= 30:                                   # sirf title wale tukde skip
            return
        if parent.startswith("7."):                           # FAQ: har Q/A alag chunk
            for qa in re.split(r"\n(?=Q:)", body):
                if len(qa.strip()) > 30:
                    chunks.append({"heading": parent, "text": qa.strip()})
        else:
            chunks.append({"heading": heading, "text": body})

    for line in text.splitlines():
        if SECTION.match(line):
            flush()
            parent = heading = line.strip()
            continue
        if BRANCH.match(line):
            flush()
            heading = f"{parent} > {line.strip()}"            # parent section bhi saath rakho
            continue
        buf.append(line)
    flush()
    return chunks

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, c in enumerate(chunk_by_section(doc["text"])):
            chunks.append({
                "id": f"{doc['source']}-{i}",
                "source": doc["source"],
                "heading": c["heading"],
                "text": c["text"],
            })
    return chunks

chunks = chunk_documents(load_documents("data.txt"))
print(len(chunks), "chunks")
print(json.dumps(chunks[:3], indent=2, ensure_ascii=False))   # pehle 3 hi dekho

21 chunks
[
  {
    "id": "data.txt-0",
    "source": "data.txt",
    "heading": "",
    "text": "﻿Xander Restaurant --- Knowledge Base\nDocument type: Synthetic demo data for RAG testing\nVersion: 1.0\nImportant: Xander Restaurant and all details in this document are fictional sample data created for testing. Replace them with verified business information before using this knowledge base with real customers."
  },
  {
    "id": "data.txt-1",
    "source": "data.txt",
    "heading": "1. Restaurant Overview",
    "text": "Name: Xander Restaurant\nCuisine: Casual dining with a mix of burgers, sandwiches, rice dishes, grilled items, and local favourites\nPositioning: A friendly, modern restaurant focused on freshly prepared meals, approachable prices, and convenient dining.\nService options: Dine-in, takeaway, and delivery (availability may vary by branch).\nCurrency: Pakistani Rupees (PKR / Rs.)\nXander Restaurant aims to make everyday dining convenient for families, students, office wo

In [6]:
import chromadb

db = chromadb.PersistentClient(path="company_db")           # saved in the folder "company_db": not lost
store = db.get_or_create_collection(
    name="xanders_data",
    metadata={
        "hnsw:space": "cosine"
    }
)

if store.count() == 0:                                      # embed only the first time (saves free requests)
    ids = []
    documents = []
    metadatas = []
    for c in chunks:
        ids.append(c["id"])
        documents.append(c["text"])
        metadatas.append({      # keep the file name with each chunk
            "source": c["source"]
        })

    store.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embed(documents),
    )
print(store.count(), "chunks in the database")

21 chunks in the database


In [7]:
def retrieve(question, n=3):
    r = store.query(query_embeddings=embed([question]), n_results=n)
    texts = r["documents"][0]          # [0] = results for our first (only) question
    sources = r["metadatas"][0]
    distances = r["distances"][0]

    found = []
    for i in range(len(texts)):
        found.append({
            "text": texts[i],
            "source": sources[i]["source"],
            "distance": distances[i]
        })
    return found
# for f in retrieve("where is your main branch"):
#     print(f"{f['distance']:.2f}  [{f['source']}]  {f['text'][:70]}")

In [8]:
RAG_SYSTEM = """You are the Customer support bot of a Xander Restaurant.
Answer the customer's question using the context.
Be polite, and answer in respectable manners . Reply in at most 3 short sentences."""

def rag(question):
    found = retrieve(question)                                          # 1. Retrieve
    pieces = []
    for f in found:
        pieces.append(f["text"])
    context = "\n\n".join(pieces)
    messages = [        # 2. Augment
        {
            "role": "system",
            "content": RAG_SYSTEM
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]
    return ask(messages)

In [9]:
import pprint
pprint.pprint(rag("where is the flagship branch"))

('The flagship branch is the Gulshan-e-Iqbal location (XAN-KHI-01), situated '
 'at Block 13-D, University Road, Gulshan-e-Iqbal, Karachi. It is the original '
 'Xander outlet and the only branch serving exclusive dishes like the Gulshan '
 'Special Beef Burger. Please let me know if you need any further assistance!')


In [10]:
# pprint.pprint(rag("When was Xander Restaurant founded?"))
print(rag("When was Xander Restaurant founded?").count)

<built-in method count of str object at 0x00000250A55C42B0>


In [11]:
pprint.pprint(rag("Dish Available branch Branch code"))

("I'd be happy to help! Could you please specify which dish you're asking "
 "about? I'll then let you know which branch offers it and confirm current "
 'availability.')


In [12]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [13]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [14]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [15]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    while True:
        completion = client.chat.completions.create(
                model=MODEL, 
                messages=messages, 
                tools=tools,
                temperature=0
            )
           
        reply = completion.choices[0].message
        messages.append(reply)
    
        # reply_json = reply.model_dump(exclude_none=True)
        # reply_json.pop("reasoning", None)            # hide the model's long thinking
        # reply_json.pop("reasoning_details", None)
        # print("LLM REPLY:", json.dumps(reply_json, indent=2))
    
        if not reply.tool_calls: 
            return Markdown(reply.content)
            break                        # model asked for a tool
       
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})        

In [16]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

Tool called: validate_order_id {'order_id': 'OR-1045'} True
Tool called: log_complaint {'customer_name': 'Sam', 'phone': '0322342423', 'complaint': 'Roll served instead of burger', 'order_id': 'OR-1045'} {'message': 'Saved! Total Complaint: 1'}


I’m really sorry about the mix‑up with your order, Sam! I’ve logged your complaint (OR‑1045) and our team will look into it right away. Thank you for bringing this to our attention—we appreciate you helping us make Xander Restaurant better. 🙏

In [17]:
print(json.dumps(complaint_list))

[{"customer_name": "Sam", "order_id": "OR-1045", "phone": "0322342423", "complaint": "Roll served instead of burger", "status": "Pending"}]




Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID